# EDA — Bank Credit Card Fraud Data

**Dataset**: `creditcard.csv`  
**Target**: `Class` (1 = Fraud, 0 = Legitimate)  
**Note**: V1–V28 are PCA-transformed anonymized features. Only `Time` and `Amount` are original.

---

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from data_loader import load_creditcard, data_quality_report
from preprocessing import clean_creditcard
from visualization import (
    plot_class_distribution,
    plot_numerical_distributions,
    plot_bivariate,
    plot_correlation_heatmap,
)

os.makedirs('../notebooks/plots', exist_ok=True)
print('✅ Imports successful')

## 1. Load & Inspect Raw Data

In [ ]:
cc_raw = load_creditcard('../data/raw/creditcard.csv')
cc_raw.head()

In [ ]:
quality = data_quality_report(cc_raw, 'CreditCard (Raw)')

## 2. Data Cleaning

In [ ]:
cc_clean = clean_creditcard(cc_raw)
print(f'Rows removed: {len(cc_raw) - len(cc_clean):,}')
data_quality_report(cc_clean, 'CreditCard (Cleaned)')

## 3. Class Imbalance Analysis

> Credit card fraud datasets are notoriously imbalanced — typically <1% fraud rate.
> This makes accuracy completely misleading as a metric.

In [ ]:
class_counts = cc_clean['Class'].value_counts()
print('Class Distribution:')
print(f'  Legitimate (0): {class_counts.get(0,0):,}  ({class_counts.get(0,0)/len(cc_clean)*100:.3f}%)')
print(f'  Fraud      (1): {class_counts.get(1,0):,}  ({class_counts.get(1,0)/len(cc_clean)*100:.3f}%)')
print(f'  Imbalance ratio: {class_counts.get(0,1)/class_counts.get(1,1):.0f}:1')

plot_class_distribution(cc_clean['Class'], 'Credit Card Fraud')

## 4. Amount and Time Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amount distribution
for cls, label, color in [(0, 'Legitimate', '#2ecc71'), (1, 'Fraud', '#e74c3c')]:
    subset = cc_clean[cc_clean['Class'] == cls]['Amount']
    axes[0].hist(subset.clip(upper=subset.quantile(0.99)), bins=50,
                 alpha=0.6, label=label, color=color)
axes[0].set_title('Transaction Amount Distribution by Class', fontweight='bold')
axes[0].set_xlabel('Amount ($)')
axes[0].legend()

# Time distribution
for cls, label, color in [(0, 'Legitimate', '#2ecc71'), (1, 'Fraud', '#e74c3c')]:
    subset = cc_clean[cc_clean['Class'] == cls]['Time'] / 3600  # convert to hours
    axes[1].hist(subset, bins=48, alpha=0.6, label=label, color=color)
axes[1].set_title('Transaction Time Distribution (Hours)', fontweight='bold')
axes[1].set_xlabel('Hours since first transaction')
axes[1].legend()

plt.suptitle('Amount and Time Analysis — Credit Card Fraud', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/plots/cc_amount_time_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Amount statistics by class
print('Amount statistics by class:')
cc_clean.groupby('Class')['Amount'].describe().round(2)

## 5. PCA Feature Distributions (V1–V28)

In [ ]:
# Distribution of PCA features: compare fraud vs. legit
# Show top 8 features that differ most between classes
v_cols = [f'V{i}' for i in range(1, 29)]
diff_means = abs(
    cc_clean[cc_clean['Class'] == 1][v_cols].mean() -
    cc_clean[cc_clean['Class'] == 0][v_cols].mean()
).sort_values(ascending=False)

top_diff_cols = diff_means.head(8).index.tolist()
print(f'Top features by mean difference between classes: {top_diff_cols}')

plot_bivariate(cc_clean, top_diff_cols[:6], 'Class', 'Credit Card (Top PCA Features)')

## 6. Correlation Analysis

In [ ]:
# Correlation with target
corr_with_target = cc_clean.corr()['Class'].drop('Class').abs().sort_values(ascending=False)
print('Top 10 features correlated with fraud:')
print(corr_with_target.head(10))

fig, ax = plt.subplots(figsize=(10, 5))
corr_with_target.head(15).plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Feature Correlation with Fraud (Class)', fontsize=12, fontweight='bold')
ax.set_ylabel('|Correlation|')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.savefig('../notebooks/plots/cc_feature_target_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Cleaned Dataset

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
cc_clean.to_csv('../data/processed/creditcard_cleaned.csv', index=False)
print(f'✅ Saved creditcard_cleaned.csv — shape: {cc_clean.shape}')

## 8. EDA Summary

| Finding | Detail |
|---------|--------|
| Class imbalance | <0.2% fraud — extreme imbalance |
| High-signal PCA features | V4, V11, V12, V14, V17 show large mean shifts |
| Amount | Fraud transactions often involve smaller, targeted amounts |
| Time | No strong temporal pattern — fraud distributed throughout |

> **Next Step**: Modeling notebook → `modeling.ipynb`